# GRPO: депрессивный стиль эссе (Qwen3-4B + depression_reward)

Пайплайн: установка → self-check reward-пакета → **baseline** (эссе instruct-моделью без обучения) → **калибровка** reward → **GRPO** (LoRA) → сравнение маркеров стиля.

**Перед запуском:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Загрузите `depression_reward.zip` в файлы Colab (создаётся в корне репозитория: `zip -r depression_reward.zip depression_reward`).
3. Runtime → Run all. Обучение (~100 шагов) занимает порядка 1–2 часов на T4.

**Приватность:** пакет содержит код TITANIS — ноутбук и файлы не публиковать, доступ по ссылке не открывать.

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"  # нужно isanlp внутри reward-пакета
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
# фикс из туториала: эти пакеты конфликтуют в Colab
!pip uninstall torchcodec sentence-transformers -y

### Reward-пакет

Распаковываем `depression_reward.zip`, ставим его зависимости и гоняем self-check — все 9 проверок должны быть PASS (заодно докачается mystem-бинарник и, при несовпадении версии sklearn, модель пересоберётся из весов).

In [ ]:
# [embed_zip_in_notebook.py] depression_reward.zip → base64, не редактировать вручную
import base64, hashlib, os, shutil
_SHA = 'ab328905f94c1fc8097ee8496678fcbeb4eb35b8bca4e682676df5c4b64fd388'
_have = os.path.exists('depression_reward.zip') and hashlib.sha256(
    open('depression_reward.zip', 'rb').read()).hexdigest() == _SHA
if _have:
    print('актуальный depression_reward.zip уже в рантайме')
else:
    _B64 = (
"..."
    )
    _data = base64.b64decode(_B64)
    assert hashlib.sha256(_data).hexdigest() == _SHA, 'повреждён встроенный zip'
    with open('depression_reward.zip', 'wb') as _f:
        _f.write(_data)
    if os.path.isdir('depression_reward'):
        shutil.rmtree('depression_reward')  # устаревшая распаковка, пересоздастся ниже
    print('depression_reward.zip обновлён из ноутбука:', len(_data), 'байт')


In [ ]:
import os, zipfile
assert os.path.isdir('depression_reward') or os.path.exists('depression_reward.zip'), \
    'Загрузите depression_reward.zip в файлы Colab (панель слева)'
if not os.path.isdir('depression_reward'):
    with zipfile.ZipFile('depression_reward.zip') as z:
        z.extractall('.')
!uv pip install -qqq -r depression_reward/requirements-colab.txt

In [ ]:
!python -m depression_reward.selfcheck --timing

### Модель

Qwen3-4B-Instruct-2507 (без `<think>`-режима; адаптер reward всё равно вырезает `<think>`, так что можно подставить и обычный Qwen3-4B или 7B — смена модели = одна строка `model_name`).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # промпты короткие, почти всё уходит на эссе
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",  # <- смена базовой модели здесь
    max_seq_length = max_seq_length,
    load_in_4bit = False,   # поставить True, если OOM на T4
    fast_inference = True,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### Данные

80 нейтральных тем из пакета; последние 10 держим как eval-набор (модель их не видит при обучении).

In [ ]:
import json
from datasets import Dataset

with open('depression_reward/prompts/prompts.jsonl') as f:
    all_prompts = [json.loads(line)['prompt'] for line in f if line.strip()]

EVAL_N = 10
train_prompts = all_prompts[:-EVAL_N]
eval_prompts  = all_prompts[-EVAL_N:]

train_dataset = Dataset.from_list(
    [{'prompt': [{'role': 'user', 'content': p}]} for p in train_prompts])
print(len(train_prompts), 'train /', len(eval_prompts), 'eval')
train_dataset[0]

### Baseline — точка отсчёта

Генерируем ~30 эссе необученной моделью. По ним: (а) средний raw-скор классификатора → **калибровка `style_center`** (иначе сигмоида style насытится и GRPO не получит сигнала); (б) стартовые Depression Markers.

In [ ]:
from vllm import SamplingParams

def generate_essays(prompts, lora_request=None, temperature=0.8,
                    max_tokens=1300, seed=None):
    texts = [tokenizer.apply_chat_template(
                 [{'role': 'user', 'content': p}],
                 tokenize=False, add_generation_prompt=True)
             for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=0.95,
                        max_tokens=max_tokens, seed=seed)
    outs = model.fast_generate(texts, sampling_params=sp,
                               lora_request=lora_request)
    return [o.outputs[0].text for o in outs]

N_BASELINE = 30
baseline_texts = generate_essays(train_prompts[:N_BASELINE], seed=3407)
print(baseline_texts[0][:600])

In [ ]:
import numpy as np
from depression_reward import DepressionReward, RewardConfig

rm_default = DepressionReward()
bd = rm_default.breakdown(baseline_texts)
valid = [b for b in bd if not b['floored']]
print(f'валидных: {len(valid)}/{len(bd)}')
assert valid, 'все baseline-тексты зафлорены — проверьте генерацию выше'

raws = [b['raw_score'] for b in valid]
style_center = float(np.mean(raws))
# temp ~ std/2: сигмоида покрывает реальный разброс скоров, а не ступенька
style_temp = round(float(np.std(raws)) / 2, 3)
print(f'style_center (калиброванный): {style_center:.4f}, style_temp: {style_temp}')
print(f"средние по baseline: reward={np.mean([b['reward'] for b in bd]):.3f}, "
      f"слов={np.mean([b['word_count'] for b in valid]):.0f}, "
      f"antisem_rate={np.mean([b['antisem_rate'] for b in valid]):.4f}")

with open('baseline_essays.jsonl', 'w') as f:
    for p, t, b in zip(train_prompts[:N_BASELINE], baseline_texts, bd):
        f.write(json.dumps({'prompt': p, 'completion': t,
                            'reward': b['reward'], 'raw_score': b['raw_score'],
                            'word_count': b['word_count'], 'floored': b['floored']},
                           ensure_ascii=False) + '\n')


In [ ]:
from depression_reward.validation import markers_report
print('=== Маркеры baseline (цель: после GRPO грамматика уйдёт к «депрессии», sentiment останется нейтральным) ===')
print(markers_report([t for t, b in zip(baseline_texts, bd) if not b['floored']]))

### GRPO

Reward — калиброванный `depression_style_reward`; breakdown каждого шага пишется в `reward_log.jsonl` (следите, за счёт чего растёт reward: style или обход штрафов).

In [ ]:
from depression_reward import make_reward_func

cfg = RewardConfig(style_center=style_center, style_temp=style_temp)
reward_func = make_reward_func(cfg, log_path='reward_log.jsonl')


In [ ]:
lens = [len(tokenizer.apply_chat_template([{'role': 'user', 'content': p}],
                                          add_generation_prompt=True, tokenize=True))
        for p in train_prompts]
max_prompt_length = max(lens) + 1
max_completion_length = max_seq_length - max_prompt_length
print('max_prompt_length =', max_prompt_length,
      '| max_completion_length =', max_completion_length)

vllm_sampling_params = SamplingParams(
    min_p = 0.1, top_p = 1.0, top_k = -1, seed = 3407,
    stop = [tokenizer.eos_token], include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 1e-5,          # 5e-6 за 100 шагов почти не сдвинул политику (KL ~0.001)
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,
    num_generations = 4,        # уменьшить, если OOM
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 300,            # ~5.5 ч на T4; если к шагу 150 KL < 0.01 — поднять LR до 2e-5
    save_steps = 100,
    report_to = "none",
    output_dir = "outputs",
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [reward_func],
    args = training_args,
    train_dataset = train_dataset,
)
trainer.train()

In [ ]:
model.save_lora("grpo_depression_lora")

### Оценка: baseline vs GRPO на отложенных промптах

Успех = средний raw/style вырос, грамматические маркеры сдвинулись к «депрессии», а `sentiment` и `antisem_rate` НЕ ушли в депрессивную лексику (иначе модель выучила семантику, а не стиль).

In [ ]:
EVAL_SEEDS = [3407, 3408, 3409]  # 10 промптов x 3 сида = 30 эссе на сторону

def generate_eval(lora_request=None):
    texts = []
    for s in EVAL_SEEDS:
        texts += generate_essays(eval_prompts, lora_request=lora_request, seed=s)
    return texts

baseline_eval = generate_eval()
trained_eval  = generate_eval(model.load_lora("grpo_depression_lora"))

rm_cal = DepressionReward(cfg)

def summarize(name, texts):
    b = rm_cal.breakdown(texts)
    ok = [x for x in b if not x['floored']]
    print(f"{name:16s} reward={np.mean([x['reward'] for x in b]):.3f}  "
          f"raw={np.mean([x['raw_score'] for x in ok]):.4f}  "
          f"style={np.mean([x['style'] for x in ok]):.3f}  "
          f"antisem={np.mean([x['antisem_rate'] for x in ok]):.4f}  "
          f"слов={np.mean([x['word_count'] for x in ok]):.0f}  "
          f"floored={len(b) - len(ok)}")
    return b

bd_base = summarize('baseline (eval)', baseline_eval)
bd_grpo = summarize('GRPO (eval)', trained_eval)

raw_b = [x['raw_score'] for x in bd_base if not x['floored']]
raw_g = [x['raw_score'] for x in bd_grpo if not x['floored']]
vb, vg = np.var(raw_b, ddof=1) / len(raw_b), np.var(raw_g, ddof=1) / len(raw_g)
t = (np.mean(raw_g) - np.mean(raw_b)) / np.sqrt(vb + vg)
print(f"\nΔraw = {np.mean(raw_g) - np.mean(raw_b):+.4f}, Welch t = {t:.2f}  (|t| > ~2 — сдвиг значим)")

print('\n=== Маркеры baseline (eval) ===')
print(markers_report(baseline_eval))
print('\n=== Маркеры GRPO (eval) ===')
print(markers_report(trained_eval))


In [ ]:
print('=== Пример эссе после GRPO ===')
print(trained_eval[0][:1500])

pairs = [(p, s) for s in EVAL_SEEDS for p in eval_prompts]  # порядок как в generate_eval
with open('grpo_eval_essays.jsonl', 'w') as f:
    for (p, s), t, b in zip(pairs, trained_eval, bd_grpo):
        f.write(json.dumps({'prompt': p, 'seed': s, 'completion': t,
                            'reward': b['reward'], 'raw_score': b['raw_score'],
                            'floored': b['floored']}, ensure_ascii=False) + '\n')
with open('baseline_eval_essays.jsonl', 'w') as f:
    for (p, s), t, b in zip(pairs, baseline_eval, bd_base):
        f.write(json.dumps({'prompt': p, 'seed': s, 'completion': t,
                            'reward': b['reward'], 'raw_score': b['raw_score'],
                            'floored': b['floored']}, ensure_ascii=False) + '\n')


### Сохранение артефактов

Скачайте `grpo_artifacts.zip` (панель «Файлы») или раскомментируйте копирование в Drive — сессия Colab всё сотрёт.

In [ ]:
!zip -q -r grpo_artifacts.zip grpo_depression_lora baseline_essays.jsonl grpo_eval_essays.jsonl baseline_eval_essays.jsonl reward_log.jsonl
!ls -lh grpo_artifacts.zip

# from google.colab import drive
# drive.mount('/content/drive')
# !cp grpo_artifacts.zip /content/drive/MyDrive/